# reidfm: train on Kaggle (2x T4)

Settings: **GPU T4 x2**, Internet **on**.

Inputs to attach:
- the pool datasets built by `00_prepare_pool.ipynb` (each holds `<key>/manifest.csv` folders);
- session 2+: the previous version's output of this notebook, to resume.

The run stops by itself before the 12 h limit (`TIME_BUDGET_H`) with a checkpoint. Save a
version, attach its output as an input, set `RESTORE_FROM`, and run again.

In [ ]:
import os, time
os.environ["REIDFM_SESSION_START"] = str(time.time())  # the time guard counts from here

# ---- edit these ----
REPO_URL = "https://github.com/<you>/Person-Re-ID-Foundational-model.git"
CONFIG = "configs/finetune_sbs_r50ibn_msmt.yml"   # or finetune_sbs_r101ibn_msmt.yml / scratch_sbs_r50ibn_imagenet.yml
RUN_NAME = "sbs_r50ibn_msmt_v1"
RESTORE_FROM = ""        # session 2+: e.g. "/kaggle/input/reidfm-train-s1"
TIME_BUDGET_H = 11.0
NUM_GPUS = 2
# --------------------

os.environ["REIDFM_DATA_ROOT"] = "/kaggle/input"
os.environ["REIDFM_OUTPUT_ROOT"] = "/kaggle/working/output"
os.environ["REIDFM_WEIGHTS_ROOT"] = "/kaggle/working/weights"
os.environ["REIDFM_TIME_BUDGET_H"] = str(TIME_BUDGET_H)

In [ ]:
%cd /kaggle/working
!test -d reidfm-repo || git clone -q $REPO_URL reidfm-repo
%cd /kaggle/working/reidfm-repo
!git pull -q
!python scripts/setup_fastreid.py --cython
!pip install -q -e ".[train]"
!python scripts/list_models.py --init reid_pretrained | head -20

In [ ]:
# Sanity check: every dataset the config needs is attached.
import sys
sys.path.insert(0, ".")
from reidfm.config import load_cfg
from reidfm.data.manifest import find_manifest
from reidfm.data.pool import load_pool

pool = load_pool(load_cfg(CONFIG).REIDFM.POOL)
missing = []
for key in pool.all_datasets + pool.val_sets:
    try:
        print(f"{key:14s} {find_manifest(key)}")
    except FileNotFoundError:
        missing.append(key)
print("missing:", missing or "none")
assert not set(missing) & set(pool.sources + pool.val_sets), "attach the missing source/val datasets"

In [ ]:
# Train with DDP. --resume is a no-op on a fresh run; --restore-from copies the previous session in first.
restore = f"--restore-from {RESTORE_FROM}" if RESTORE_FROM else ""
!python scripts/train.py --config-file $CONFIG --num-gpus $NUM_GPUS --output-dir $RUN_NAME --resume $restore

In [ ]:
# Where the run stands. The output folder is what the next session restores.
run_dir = f"/kaggle/working/output/{RUN_NAME}"
!cat $run_dir/last_checkpoint; echo
!ls -lh $run_dir | grep -E "pth|yaml|json"
!grep -E "val |Time guard|Resumed" $run_dir/log.txt | tail -20
# Keep the saved output small: weights and the code checkout are re-created each session.
!rm -rf /kaggle/working/weights /kaggle/working/reidfm-repo